In [12]:
from pathlib import Path
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
import json
import random
import time
from typing import Any

import pandas as pd
import requests
import matplotlib.pyplot as plt


BASE_URL = "https://dummyjson.com"

DATA_DIR = Path(".")
RAW_PATH = DATA_DIR / "data_raw.jsonl"
CLEAN_PATH = DATA_DIR / "data_clean.csv"
LOG_PATH = DATA_DIR / "collection_log.csv"
STATE_PATH = DATA_DIR / "state.json"

REQUEST_TIMEOUT = 15
MIN_REQUEST_INTERVAL = 0.25

MAX_RETRIES = 3
BACKOFF_BASE = 1.0
JITTER_MAX = 0.5

In [2]:
class RateLimiter:
    def __init__(self, min_interval: float = 0.25):
        self.min_interval = min_interval
        self.last_request_time = 0.0
 
    def wait(self):
        now = time.monotonic() # фиксируем текущее время 
        elapsed = now - self.last_request_time # Считаем, сколько прошло с прошлого действия, т.е интервал на текущий запрос

        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)

        self.last_request_time = time.monotonic()

In [13]:
# Api клиент с логами 

class APIClient:
    def __init__(
        self,
        base_url: str,
        timeout: int = 15,
        min_interval: float = 0.25,
        max_retries: int = 3,
    ):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.max_retries = max_retries

        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "lab3-rest-api/1.0",
            "Accept": "application/json",
        })

        self.rate_limiter = RateLimiter(min_interval)

        self.logs = []

    @staticmethod # сообщаем сколько секунд нужно подождать перед повторной попыткой запроса
    def _retry_after_seconds(value: str | None) -> float | None:
 
        if not value:
            return None

        value = value.strip()

        # Вариант: Retry-After: 5
        try:
            return max(0.0, float(value))
        except ValueError:
            pass

        # Вариант: Retry-After: Wed, 21 Oct 2015 07:28:00 GMT
        try:
            retry_dt = parsedate_to_datetime(value)

            if retry_dt.tzinfo is None:
                retry_dt = retry_dt.replace(tzinfo=timezone.utc)

            now = datetime.now(timezone.utc)

            return max(
                0.0,
                (retry_dt - now).total_seconds()
            )

        except (TypeError, ValueError, OverflowError):
            return None

    @staticmethod  # считает сколько записей пришло в ответе от API
    def _records_count(data: Any, items_key: str | None) -> int | None:
        if items_key is None:
            return None

        if not isinstance(data, dict):
            return None

        items = data.get(items_key)

        if isinstance(items, list):
            return len(items)

        return None

    def get_json(
        self,
        endpoint: str,
        params: dict | None = None,
        *,
        source: str,
        page: int | None = None,
        items_key: str | None = None,
    ) -> dict:
        """
        Безопасный GET-запрос.

        Повторы разрешены только для:
        - 429
        - 5xx

        Для 4xx повторов нет.
        """

        url = f"{self.base_url}/{endpoint.lstrip('/')}"

        # цикл повторных попыток
        for attempt in range(1, self.max_retries + 2):

            self.rate_limiter.wait()

            started = time.perf_counter()
            timestamp = datetime.now(timezone.utc).isoformat()

            try:
                response = self.session.get(
                    url,
                    params=params,
                    timeout=self.timeout,
                )
            # логи с проблемой
            except requests.RequestException as exc:

                duration = time.perf_counter() - started

                self.logs.append({
                    "timestamp": timestamp,
                    "source": source,
                    "page": page,
                    "url": url,
                    "params": json.dumps(
                        params or {},
                        ensure_ascii=False,
                        sort_keys=True,
                    ),
                    "status_code": None,
                    "records": None,
                    "duration_sec": round(duration, 4),
                    "attempt": attempt,
                    "error": str(exc),
                })

                # если выпала сетевая ошибка
                # не ретраим: повторы только 429 и 5xx.
                raise RuntimeError(
                    f"Ошибка сети для {url}: {exc}"
                ) from exc

            duration = time.perf_counter() - started

            status = response.status_code

            # Успешный ответ

            if 200 <= status < 300:

                # проверяем header
                content_type = response.headers.get(
                    "Content-Type",
                    ""
                ).lower()

                if "application/json" not in content_type:
                    body_preview = response.text[:200]

                    self.logs.append({
                        "timestamp": timestamp,
                        "source": source,
                        "page": page,
                        "url": response.url,
                        "params": json.dumps(
                            params or {},
                            ensure_ascii=False,
                            sort_keys=True,
                        ),
                        "status_code": status,
                        "records": None,
                        "duration_sec": round(duration, 4),
                        "attempt": attempt,
                        "error": (
                            f"Ожидался JSON, получен "
                            f"{content_type}. "
                            f"Тело: {body_preview}"
                        ),
                    })

                    raise RuntimeError(
                        f"API вернул не JSON: {content_type}. "
                        f"Первые 200 байт: {body_preview}"
                    )

                # проверяем валидность json
                try:
                    data = response.json()
                except ValueError as exc:

                    body_preview = response.text[:200]

                    self.logs.append({
                        "timestamp": timestamp,
                        "source": source,
                        "page": page,
                        "url": response.url,
                        "params": json.dumps(
                            params or {},
                            ensure_ascii=False,
                            sort_keys=True,
                        ),
                        "status_code": status,
                        "records": None,
                        "duration_sec": round(duration, 4),
                        "attempt": attempt,
                        "error": (
                            f"Некорректный JSON: {body_preview}"
                        ),
                    })

                    raise RuntimeError(
                        f"Некорректный JSON. "
                        f"Первые 200 байт: {body_preview}"
                    ) from exc

                records_count = self._records_count(
                    data,
                    items_key
                )

                self.logs.append({
                    "timestamp": timestamp,
                    "source": source,
                    "page": page,
                    "url": response.url,
                    "params": json.dumps(
                        params or {},
                        ensure_ascii=False,
                        sort_keys=True,
                    ),
                    "status_code": status,
                    "records": records_count,
                    "duration_sec": round(duration, 4),
                    "attempt": attempt,
                    "error": None,
                })

                return data

            # Обработка временных ошибок
            # Retry: 429 / 5xx
            # 429 (Спам запросов) и 5xx (Сервер упал)

            retryable = status == 429 or 500 <= status <= 599

            if retryable and attempt <= self.max_retries:

                retry_after = self._retry_after_seconds(
                    response.headers.get("Retry-After")
                )
                # Экспоненциальное увеличение задержки с каждой попыткой
                backoff = (
                    BACKOFF_BASE * (2 ** (attempt - 1))
                )
                # случайная небольшая прибавка, чтобы запросы не отправлялись одновременно
                jitter = random.uniform(
                    0,
                    JITTER_MAX
                )

                sleep_seconds = backoff + jitter

                if retry_after is not None:
                    sleep_seconds = max(
                        sleep_seconds,
                        retry_after
                    )

                body_preview = response.text[:200]

                self.logs.append({
                    "timestamp": timestamp,
                    "source": source,
                    "page": page,
                    "url": response.url,
                    "params": json.dumps(
                        params or {},
                        ensure_ascii=False,
                        sort_keys=True,
                    ),
                    "status_code": status,
                    "records": None,
                    "duration_sec": round(duration, 4),
                    "attempt": attempt,
                    "error": (
                        f"Retry через {sleep_seconds:.2f} сек; "
                        f"body={body_preview}"
                    ),
                })

                time.sleep(sleep_seconds)
                continue

            # ==================================================
            # Остальные ошибки

            body_preview = response.text[:200]

            self.logs.append({
                "timestamp": timestamp,
                "source": source,
                "page": page,
                "url": response.url,
                "params": json.dumps(
                    params or {},
                    ensure_ascii=False,
                    sort_keys=True,
                ),
                "status_code": status,
                "records": None,
                "duration_sec": round(duration, 4),
                "attempt": attempt,
                "error": (
                    f"HTTP {status}: {body_preview}"
                ),
            })

            # Важно: 4xx здесь не ретраится.
            raise requests.HTTPError(
                f"HTTP {status} для {response.url}. "
                f"Первые 200 байт: {body_preview}",
                response=response,
            )

        raise RuntimeError(
            f"Не удалось получить {url} после "
            f"{self.max_retries + 1} попыток."
        )

    # логи записываем в DataFrame
    def get_log_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.logs)

In [14]:
def paginate(
    client: APIClient,
    endpoint: str,
    items_key: str,
    page_size: int,
    max_records: int,
    max_pages: int,
    start_skip: int = 0,
    extra_params: dict | None = None,
):
    """
    Универсальная пагинация через limit/skip.

    Останавливаемся при:
    - пустой странице;
    - достижении total;
    - достижении max_records;
    - достижении max_pages.
    """

    records = []

    skip = start_skip

    # Номер реальной страницы при page_size=100:
    # skip=0   -> 1
    # skip=100 -> 2
    # skip=200 -> 3
    # skip=300 -> 4
    page_number = start_skip // page_size + 1

    for _ in range(max_pages):

        remaining = max_records - len(records)

        if remaining <= 0:
            break

        current_limit = min(
            page_size,
            remaining
        )

        params = {
            "limit": current_limit,
            "skip": skip,
        }

        if extra_params:
            params.update(extra_params)

        response_data = client.get_json(
            endpoint,
            params=params,
            source=endpoint.strip("/"),
            page=page_number,
            items_key=items_key,
        )

        items = response_data.get(items_key)

        if not isinstance(items, list):
            raise ValueError(
                f"Ответ {endpoint} не содержит "
                f"список {items_key}."
            )

        actual_len = len(items)

        records.extend(items)

        # ВАЖНО:
        # увеличиваем skip на фактическую длину ответа.
        skip += actual_len
        page_number += 1

        # Пустая страница
        if actual_len == 0:
            break

        total = response_data.get("total")

        # Достигли общего количества записей
        if total is not None and skip >= int(total):
            break

    return records

In [15]:
""" 
функции отвечают за дедупликацию и сохранение данных в файл формата JSONL (JSON Lines, где каждая строчка — это отдельный JSON-объект).
Простыми словами: они позволяют скачивать новые данные и дозаписывать их в файл так, чтобы не дублировать записи, 
которые вы уже скачали ранее (например, при повторном запуске скрипта или если с сервера пришла та же страница). 
"""
def load_raw_keys(path: Path) -> set[tuple[str, int]]:
    seen = set()

    if not path.exists():
        return seen

    with path.open(
        "r",
        encoding="utf-8"
    ) as f:

        for line_number, line in enumerate(f, start=1):

            line = line.strip()

            if not line:
                continue

            try:
                item = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Некорректная JSONL-строка "
                    f"{line_number}: {exc}"
                ) from exc

            source = item.get("source")
            record = item.get("record", {})
            record_id = record.get("id")

            if source is not None and record_id is not None:
                seen.add(
                    (
                        str(source),
                        int(record_id),
                    )
                )

    return seen


def append_raw_records(
    path: Path,
    records: list[dict],
    source: str,
    seen_keys: set[tuple[str, int]],
):
    if not records:
        return 0

    written = 0

    with path.open(
        "a",
        encoding="utf-8"
    ) as f:

        retrieved_at = datetime.now(
            timezone.utc
        ).isoformat()

        for record in records:

            record_id = record.get("id")

            if record_id is None:
                # Для нашей задачи ID обязателен.
                continue

            key = (
                str(source),
                int(record_id),
            )

            if key in seen_keys:
                continue

            payload = {
                "source": source,
                "retrieved_at": retrieved_at,
                "record": record,
            }

            f.write(
                json.dumps(
                    payload,
                    ensure_ascii=False
                )
                + "\n"
            )

            seen_keys.add(key)
            written += 1

    return written

In [16]:
def load_state(path: Path) -> dict:
    if not path.exists():
        return {
            "watermark": 0,
            "stored_records": 0,
            "updated_at": None,
        }

    with path.open(
        "r",
        encoding="utf-8"
    ) as f:
        state = json.load(f)

    return state


def save_state(
    path: Path,
    watermark: int,
    stored_records: int,
):
    state = {
        "watermark": int(watermark),
        "stored_records": int(stored_records),
        "updated_at": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    with path.open(
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            state,
            f,
            ensure_ascii=False,
            indent=2,
        )

In [17]:
client = APIClient(
    BASE_URL,
    timeout=REQUEST_TIMEOUT,
    min_interval=MIN_REQUEST_INTERVAL,
    max_retries=MAX_RETRIES,
)

In [19]:
comments = paginate(
    client=client,
    endpoint="/comments",
    items_key="comments",
    page_size=100,
    max_records=300,
    max_pages=10,
)

len(comments)

300

In [20]:
comments[0]

{'id': 1,
 'body': 'This is some awesome thinking!',
 'postId': 242,
 'likes': 3,
 'user': {'id': 105, 'username': 'emmac', 'fullName': 'Emma Wilson'}}

In [21]:
posts = paginate(
    client=client,
    endpoint="/posts",
    items_key="posts",
    page_size=100,
    max_records=1000,
    max_pages=10,
)
len(posts)

251

In [22]:
posts[0]

{'id': 1,
 'title': 'His mother had always taught him',
 'body': "His mother had always taught him not to ever think of himself as better than others. He'd tried to live by this motto. He never looked down on those who were less fortunate or who had less money than him. But the stupidity of the group of people he was talking to made him change his mind.",
 'tags': ['history', 'american', 'crime'],
 'reactions': {'likes': 192, 'dislikes': 25},
 'views': 305,
 'userId': 121}

In [23]:
users = paginate(
    client=client,
    endpoint="/users",
    items_key="users",
    page_size=100,
    max_records=1000,
    max_pages=10,
    extra_params={
        "select": (
            "id,"
            "firstName,"
            "lastName,"
            "username,"
            "age,"
            "gender"
        ),
    },
)

print("Всего пользователей:", len(users))

Всего пользователей: 208


In [27]:
print("Комментарии:", len(comments))
print("Посты:", len(posts))
print("Пользователи:", len(users))

Комментарии: 300
Посты: 251
Пользователи: 208


In [28]:
comments[0]

{'id': 1,
 'body': 'This is some awesome thinking!',
 'postId': 242,
 'likes': 3,
 'user': {'id': 105, 'username': 'emmac', 'fullName': 'Emma Wilson'}}

In [30]:
comments_df = pd.json_normalize(comments, sep="_")

comments_df = comments_df.rename(columns={
    "id": "comment_id",
    "postId": "post_id",
    "body": "comment_body",
    "likes": "comment_likes",
    "user_id": "comment_user_id",
    "user_username": "comment_username",
    "user_fullName": "comment_full_name",
})

comments_df.head()

,comment_id,comment_body,post_id,comment_likes,comment_user_id,comment_username,comment_full_name
0,1,This is some awesome thinking!,242,3,105,emmac,Emma Wilson
1,2,What terrific math skills you're showing!,46,4,183,cameronp,Cameron Perez
2,3,You are an amazing writer!,235,2,1,emilys,Emily Johnson
3,4,Wow! You have improved so much!,31,1,89,braydenf,Brayden Fleming
4,5,Nice idea!,212,1,149,wyattp,Wyatt Perry


In [32]:
posts_df = pd.json_normalize(posts, sep="_")

posts_df = posts_df.rename(columns={
    "id": "post_id",
    "userId": "post_author_id",
    "title": "post_title",
    "body": "post_body",
    "views": "post_views",
    "reactions_likes": "post_likes",
    "reactions_dislikes": "post_dislikes",
    "tags": "post_tags"
})

posts_df.head()

,post_id,post_title,post_body,post_tags,post_views,post_author_id,post_likes,post_dislikes
0,1,His mother had always taught him,His mother had always taught him not to ever t...,"[history, american, crime]",305,121,192,25
1,2,He was an expert but not in a discipline,He was an expert but not in a discipline that ...,"[french, fiction, english]",4884,91,859,32
2,3,Dave watched as the forest burned up on the hill.,Dave watched as the forest burned up on the hi...,"[magical, history, french]",4152,16,1448,39
3,4,All he wanted was a candy bar.,All he wanted was a candy bar. It didn't seem ...,"[mystery, english, american]",4548,47,359,18
4,5,Hopes and dreams were dashed that day.,Hopes and dreams were dashed that day. It shou...,"[crime, mystery, love]",626,131,119,30


In [38]:
users_df = pd.json_normalize(users, sep="_")

users_df = users_df.rename(columns={
    "id": "author_id",
    "firstName": "author_first_name",
    "lastName": "author_last_name",
    "username": "author_username",
    "age": "author_age",
    "gender": "author_gender",
})

users_df.head()

,author_id,author_first_name,author_last_name,author_username,author_age,author_gender
0,1,Emily,Johnson,emilys,29,female
1,2,Michael,Williams,michaelw,36,male
2,3,Sophia,Brown,sophiab,43,female
3,4,James,Davis,jamesd,46,male
4,5,Emma,Miller,emmaj,31,female


In [34]:
# Выполним мерж таблиц 

enriched_df = comments_df.merge(
    posts_df,
    on="post_id",
    how="left",
    validate="many_to_one"
)

enriched_df.head()

,comment_id,comment_body,post_id,comment_likes,comment_user_id,comment_username,comment_full_name,post_title,post_body,post_tags,post_views,post_author_id,post_likes,post_dislikes
0,1,This is some awesome thinking!,242,3,105,emmac,Emma Wilson,The serene meadow stretched to the horizon,Dotted with wildflowers and bathed in golden s...,"[meadow, serene, tranquility]",1197,60,142,34
1,2,What terrific math skills you're showing!,46,4,183,cameronp,Cameron Perez,"The clowns had taken over. And yes, they were ...","The clowns had taken over. And yes, they were ...","[crime, magical, american]",3730,132,597,23
2,3,You are an amazing writer!,235,2,1,emilys,Emily Johnson,The bustling city market was a feast for the s...,"Stalls overflowed with exotic spices, fresh pr...","[city, market, senses]",3074,88,1457,15
3,4,Wow! You have improved so much!,31,1,89,braydenf,Brayden Fleming,It was just a burger.,It was just a burger. Why couldn't she underst...,"[classic, fiction, magical]",1656,168,443,21
4,5,Nice idea!,212,1,149,wyattp,Wyatt Perry,The ancient ruins whispered tales of the past,"Stone columns stood like silent sentinels, the...","[ruins, history, ancient]",3061,130,603,38


In [39]:
merge_parsing_df = enriched_df.merge(
    users_df,
    left_on="post_author_id",
    right_on="author_id",
    how="left",
    validate="many_to_one"

)

merge_parsing_df.head()

,comment_id,comment_body,post_id,comment_likes,comment_user_id,comment_username,comment_full_name,post_title,post_body,post_tags,post_views,post_author_id,post_likes,post_dislikes,author_id,author_first_name,author_last_name,author_username,author_age,author_gender
0,1,This is some awesome thinking!,242,3,105,emmac,Emma Wilson,The serene meadow stretched to the horizon,Dotted with wildflowers and bathed in golden s...,"[meadow, serene, tranquility]",1197,60,142,34,60,Lillian,Simmons,lillians,33,female
1,2,What terrific math skills you're showing!,46,4,183,cameronp,Cameron Perez,"The clowns had taken over. And yes, they were ...","The clowns had taken over. And yes, they were ...","[crime, magical, american]",3730,132,597,23,132,Zoe,Bennett,zoec,29,female
2,3,You are an amazing writer!,235,2,1,emilys,Emily Johnson,The bustling city market was a feast for the s...,"Stalls overflowed with exotic spices, fresh pr...","[city, market, senses]",3074,88,1457,15,88,Aubrey,Wagner,aubreyw,31,female
3,4,Wow! You have improved so much!,31,1,89,braydenf,Brayden Fleming,It was just a burger.,It was just a burger. Why couldn't she underst...,"[classic, fiction, magical]",1656,168,443,21,168,Luna,Perez,lunah,27,female
4,5,Nice idea!,212,1,149,wyattp,Wyatt Perry,The ancient ruins whispered tales of the past,"Stone columns stood like silent sentinels, the...","[ruins, history, ancient]",3061,130,603,38,130,Chloe,Roberts,chloep,42,female


In [40]:
tags_df = pd.json_normalize(
    posts,
    record_path="tags",
    meta=["id", "userId"],
    errors="ignore"
)

tags_df = tags_df.rename(columns={
    0: "tag",
    "id": "post_id",
    "userId": "post_author_id"
})

tags_df.head()

,tag,post_id,post_author_id
0,history,1,121
1,american,1,121
2,crime,1,121
3,french,2,91
4,fiction,2,91


In [41]:
tag_counts = (
    tags_df
    .groupby("post_id")
    .size()
    .reset_index(name="tag_count")
)
tag_counts

,post_id,tag_count
0,1,3
1,2,3
2,3,3
3,4,3
4,5,3
...,...,...
246,247,3
247,248,3
248,249,3
249,250,3


In [44]:
enriched_df = enriched_df.merge(
    tag_counts,
    on="post_id",
    how="left"
)

enriched_df.head()

,comment_id,comment_body,post_id,comment_likes,comment_user_id,comment_username,comment_full_name,post_title,post_body,post_tags,post_views,post_author_id,post_likes,post_dislikes,tag_count_x,tag_count_y,tag_count
0,1,This is some awesome thinking!,242,3,105,emmac,Emma Wilson,The serene meadow stretched to the horizon,Dotted with wildflowers and bathed in golden s...,"[meadow, serene, tranquility]",1197,60,142,34,3,3,3
1,2,What terrific math skills you're showing!,46,4,183,cameronp,Cameron Perez,"The clowns had taken over. And yes, they were ...","The clowns had taken over. And yes, they were ...","[crime, magical, american]",3730,132,597,23,3,3,3
2,3,You are an amazing writer!,235,2,1,emilys,Emily Johnson,The bustling city market was a feast for the s...,"Stalls overflowed with exotic spices, fresh pr...","[city, market, senses]",3074,88,1457,15,3,3,3
3,4,Wow! You have improved so much!,31,1,89,braydenf,Brayden Fleming,It was just a burger.,It was just a burger. Why couldn't she underst...,"[classic, fiction, magical]",1656,168,443,21,3,3,3
4,5,Nice idea!,212,1,149,wyattp,Wyatt Perry,The ancient ruins whispered tales of the past,"Stone columns stood like silent sentinels, the...","[ruins, history, ancient]",3061,130,603,38,3,3,3


In [46]:
# Оцениваем качество данных

# процент заполненных значений
completeness = (
    enriched_df.notna().mean() * 100
).round(2)

print(completeness)

comment_id           100.0
comment_body         100.0
post_id              100.0
comment_likes        100.0
comment_user_id      100.0
comment_username     100.0
comment_full_name    100.0
post_title           100.0
post_body            100.0
post_tags            100.0
post_views           100.0
post_author_id       100.0
post_likes           100.0
post_dislikes        100.0
tag_count_x          100.0
tag_count_y          100.0
tag_count            100.0
dtype: float64


In [47]:
duplicates = enriched_df["comment_id"].duplicated().sum()

print("Дубликатов comment_id:", duplicates)

Дубликатов comment_id: 0


In [48]:
duplicates = enriched_df["comment_id"].duplicated().sum()

print("Дубликатов comment_id:", duplicates)

Дубликатов comment_id: 0


In [50]:
print(enriched_df.dtypes)

numeric_columns = [
    "comment_id",
    "post_id",
    "comment_likes",
    "post_views",
    "post_likes",
    "post_dislikes",
    "tag_count",
]

print(enriched_df[numeric_columns].dtypes)

comment_id            int64
comment_body         object
post_id               int64
comment_likes         int64
comment_user_id       int64
comment_username     object
comment_full_name    object
post_title           object
post_body            object
post_tags            object
post_views            int64
post_author_id        int64
post_likes            int64
post_dislikes         int64
tag_count_x           int64
tag_count_y           int64
tag_count             int64
dtype: object
comment_id       int64
post_id          int64
comment_likes    int64
post_views       int64
post_likes       int64
post_dislikes    int64
tag_count        int64
dtype: object


In [51]:
# Проверяем выбросы (через квантили?)
 
q1 = enriched_df["post_views"].quantile(0.25)
q3 = enriched_df["post_views"].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = enriched_df[
    (enriched_df["post_views"] < lower) |
    (enriched_df["post_views"] > upper)
]

print("Выбросов по post_views:", len(outliers))

Выбросов по post_views: 0


In [52]:
# Инкрементальный повторный запуск
state = {
    "last_comment_id": int(enriched_df["comment_id"].max())
}

with open("state.json", "w", encoding="utf-8") as f:
    json.dump(state, f, ensure_ascii=False, indent=2)

In [53]:
with open("state.json", "r", encoding="utf-8") as f:
    state = json.load(f)

last_comment_id = state["last_comment_id"]

print("Последний обработанный ID:", last_comment_id)

Последний обработанный ID: 300


In [56]:
new_comments = [
    comment
    for comment in comments
    if int(comment["id"]) > last_comment_id
]
print("Новых комментариев:", len(new_comments))

Новых комментариев: 0


In [57]:
print("Всего строк:", len(enriched_df))
print(
    "Уникальных comment_id:",
    enriched_df["comment_id"].nunique()
)

Всего строк: 300
Уникальных comment_id: 300
